# 🎙️ dots.tts SOAR on Google Colab

Clone a voice from a short clip and generate narration, saved to **Google Drive** under the narration name you choose (`1.1.wav`, `1.2.wav`, …) in a folder named after your project and today's date.

**Quick start:** *Runtime → Change runtime type → **T4 GPU*** → run cell **1️⃣**, then cell **2️⃣**, then open the `*.gradio.live` link it prints.

### 🏷️ Credits & License
* 🔗 [dots.tts on GitHub](https://github.com/studio-dots-ai/dots.tts) · 🤗 [dots.tts-soar on Hugging Face](https://huggingface.co/dots-studio/dots.tts-soar)
* 📄 Model and code: [Apache License 2.0](https://github.com/studio-dots-ai/dots.tts/blob/main/LICENSE)
* 👨‍💻 Colab notebook & app: [HiranNalaka](https://github.com/hirannalaka19)

### ⚠️ Usage Disclaimer
Use of this voice cloning model is subject to strict ethical and legal standards. By using this tool, you agree **not to**:

* **Deceive or defraud** — create misleading or fraudulent content with cloned voices.
* **Impersonate** — replicate anyone's voice without their explicit permission.
* **Break the law** — use the model in any way that violates local, national, or international law.
* **Cause harm** — create offensive, defamatory, or unethical material, or spread misinformation.

> ⚖️ **Legal responsibility:** the developers disclaim all liability for misuse. **You are responsible** for making sure your use complies with all applicable laws and ethical guidelines. Clearly label AI-generated audio.

In [ ]:
#@title 1️⃣ Install dots.tts
#@markdown Takes about 2 minutes. Needs a **GPU runtime**: *Runtime → Change runtime type → T4 GPU*.
import os
from IPython.display import clear_output

%cd /content/
!rm -rf /content/Dots-Google-Colab
!GIT_TERMINAL_PROMPT=0 git clone -q https://github.com/hirannalaka19/Dots-Google-Colab.git /content/Dots-Google-Colab
if not os.path.isfile("/content/Dots-Google-Colab/setup_colab.py"):
    raise SystemExit(
        "❌ Couldn't download the code from GitHub. The repository "
        "github.com/hirannalaka19/Dots-Google-Colab must be Public "
        "(repo → Settings → General → Danger Zone → Change visibility)."
    )
%cd /content/Dots-Google-Colab
!python setup_colab.py

if os.path.exists(".install_ok"):
    clear_output()
    print("✅ Installation complete — now run cell 2️⃣")
else:
    print("❌ Installation failed — scroll up to see the error.")

In [ ]:
#@title 2️⃣ Run the app
#@markdown ### 📁 Where to save
#@markdown Every take is saved as `<narration name>.wav` (e.g. `1.1.wav`) in **`MyDrive/<drive_folder>/<project_name>_<date>/`**. You can change the project name inside the app as well.
save_to_google_drive = True #@param {type:"boolean"}
drive_folder = "Dots TTS" #@param {type:"string"}
project_name = "My Project" #@param {type:"string"}
#@markdown ### 🔒 Lock the public link (optional)
#@markdown Anyone with the `gradio.live` link can use the app. Set a password to require a login (username: `dots`). Leave empty for no login.
app_password = "" #@param {type:"string"}
#@markdown ### ⚙️ Advanced
#@markdown `auto` = float16 on a T4, bfloat16 on A100 / L4 / H100. Pick `bfloat16` if you ever get silent or broken audio.
precision = "auto" #@param ["auto", "bfloat16", "float16", "float32"]
#@markdown Faster generation on A100 / L4 / H100 after a few minutes of warm-up. Ignored on a T4.
use_torch_compile = False #@param {type:"boolean"}

import os, subprocess, sys, time

REPO_DIR = "/content/Dots-Google-Colab"
MODEL_ID = "dots-studio/dots.tts-soar"

if not os.path.exists(f"{REPO_DIR}/.install_ok"):
    raise SystemExit("❌ Run cell 1️⃣ (Install) first.")

# Optional Hugging Face token (🔑 Secrets → HF_TOKEN). The model is public;
# a token only makes the 5 GB download faster and avoids rate limits.
try:
    from google.colab import userdata
    token = (userdata.get("HF_TOKEN") or "").strip()
except Exception:
    token = ""
if token:
    os.environ["HF_TOKEN"] = token
    print("🔑 Using HF_TOKEN from Colab Secrets")
else:
    print("ℹ️ No HF_TOKEN secret — downloading anonymously (works, may be slower).")

output_root = f"{REPO_DIR}/outputs"
if save_to_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    folder = drive_folder.strip().strip("/") or "Dots TTS"
    output_root = f"/content/drive/MyDrive/{folder}"
os.makedirs(output_root, exist_ok=True)
print(f"📁 Projects are saved in {output_root}")

# Hugging Face's CDN occasionally rejects its own signed URLs (403). Files
# already downloaded are kept, so retrying only fetches what is missing: try the
# plain HTTP path a few times, then the Xet protocol path.
def download(disable_xet):
    env = dict(os.environ)
    if disable_xet:
        env["HF_HUB_DISABLE_XET"] = "1"
    else:
        env.pop("HF_HUB_DISABLE_XET", None)
    code = f"from huggingface_hub import snapshot_download; snapshot_download('{MODEL_ID}')"
    return subprocess.call([sys.executable, "-c", code], env=env) == 0

for attempt, disable_xet in enumerate([True, True, True, False, False], 1):
    print(f"⬇️ Downloading {MODEL_ID} (5.2 GB) — attempt {attempt} of 5 ...")
    if download(disable_xet):
        break
    time.sleep(5)
else:
    raise SystemExit("❌ Download failed — Hugging Face may be down (https://status.huggingface.co). Run this cell again.")
print("✅ Model ready")

os.environ.update({
    "DOTS_OUTPUT_ROOT": output_root,
    "DOTS_PROJECT": project_name,
    "DOTS_PRECISION": precision,
    "DOTS_OPTIMIZE": "1" if use_torch_compile else "0",
    "DOTS_SHARE": "1",
    "DOTS_PASSWORD": app_password,
    "HF_HUB_DISABLE_XET": "1",
})
%cd /content/Dots-Google-Colab
!python -u app.py

## 📖 Using the app

**1 · Voice** — pick one:
* **Clone: reference audio + transcript** (best) — upload 5–15 s of clean speech. Whisper writes the transcript; fix any mistakes so it matches word for word. Longer clips are cut automatically.
* **Clone: reference audio only** — copies the timbre without needing a transcript.
* **Random voice** — no reference. When you like a voice, click **📌 Use this take as the reference voice** to keep it.

**2 · Text**
* **Single narration** — type a *Narration name* such as `1.1` and the text. It is saved as `1.1.wav`, and the name moves on to `1.2` for the next take.
* **Batch script** — one narration per line, each starting with its id. Every narration is saved under its id, in one consistent voice:
  ```
  1.1: The storm had been building all afternoon.
  1.2: By nightfall, the harbour was empty.
  2.1: Morning brought an eerie calm.
  ```
  Lines without an id continue the narration above. Tick *Skip narrations already saved* to resume a batch after Colab disconnects.

**Where files go:** `MyDrive/Dots TTS/<Project name>_<YYYY-MM-DD>/1.1.wav` — the date comes from your browser. Generating the same name again replaces that file.

**Tips:** different seeds give different rhythm and intonation (the seed used is shown in *Status*); raise *Inference steps* to 16–32 for slightly better quality; tick *Also save subtitles* to get `1.1.srt`, `1.1_words.srt` and `1.1_shorts.srt` next to each take.